# Agentic RAG — Hands-On

Offline retrieve-grade-rewrite loop with deterministic tools.

## 0. Setup

In [ ]:
%pip install -q numpy
DOCS={"rag":"RAG retrieves external evidence for grounded answers.","self":"Self-RAG critiques relevance and support before answering.","crag":"CRAG corrects retrieval when evidence quality is weak.","hyde":"HyDE rewrites a query for better semantic retrieval."}
def retrieve(q):
    words=set(q.lower().split()); hits=[]
    for k,v in DOCS.items():
        score=len(words & set(v.lower().replace('.', '').split()))
        if score: hits.append((score,k,v))
    return sorted(hits, reverse=True)

## 1. Evidence grading

In [ ]:
def grade(hits):
    if len(hits)>=2 and hits[0][0]>=2: return "accept"
    if hits: return "rewrite"
    return "broaden"
for q in ["CRAG retrieval", "unknown topic"]: print(q, retrieve(q), grade(retrieve(q)))

## 2. Agent loop

In [ ]:
def rewrite(q, verdict):
    return q + (" RAG evidence retrieval" if verdict=="rewrite" else " RAG Self-RAG CRAG HyDE")

def agent(q, max_calls=3):
    trace=[]
    for i in range(max_calls):
        hits=retrieve(q); verdict=grade(hits); trace.append((q, verdict, [h[1] for h in hits]))
        if verdict=="accept": return "answer", trace
        q=rewrite(q, verdict)
    return "insufficient_context", trace
status, trace=agent("How does corrective retrieval work?")
print(status)
for t in trace: print(t)

## 3. Citation assembly

In [ ]:
last_hits=retrieve(trace[-1][0])
answer="Corrective RAG evaluates and repairs retrieval " + " ".join(f"[{k}]" for _,k,_ in last_hits[:2])
print(answer)

## 4. Exercise prompts
1. Add query de-duplication.
2. Add max latency cost.
3. Add a contradiction verdict.